In [1]:
import sys
sys.path.append("../src")
from data_loader import (load_prices, compute_hedge_ratio, compute_spread, compute_zscore, generate_signals, compute_pnl_with_costs, compute_correlation, find_cointegrated_pairs)

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

prices = load_prices()
train_prices = prices["2018-01-01" : "2021-12-31"]
test_prices = prices["2022-01-01" : "2023-12-31"]

# reload all cointegrated pairs from training data 
corr_matrix , pairs_df = compute_correlation(train_prices , threashold= 0.8)
coint_df = find_cointegrated_pairs(train_prices , pairs_df)
train_pairs = coint_df[coint_df["cointegrated"] == True].reset_index(drop = True)

print(f"Training pairs: {len(train_pairs)}")
print(train_pairs[["ticker_1" , "ticker_2" , "coint_pvalue"]].head(10))

Training pairs: 21
  ticker_1 ticker_2  coint_pvalue
0     META      TGT      0.001777
1       MA      WMT      0.003089
2        V      WMT      0.004086
3    GOOGL      UNH      0.006042
4      JNJ      TGT      0.006235
5      ABT      JNJ      0.012769
6       CL      WMT      0.013557
7      JNJ     MSFT      0.015999
8       KR      UNH      0.017131
9        C       GE      0.018694


In [2]:
# compute half-life of mean reversion   

def compute_half_life(spread):
    spread_lag = spread.shift(1).dropna()
    spread_diff = spread.diff().dropna()

    # align 
    spread_lag = spread_lag.iloc[:len(spread_diff)]
    
    # OLS regression of diff on lag 
    from statsmodels.regression.linear_model import OLS
    from statsmodels.tools import add_constant

    x = add_constant(spread_lag)
    model = OLS(spread_diff , x).fit()
    lambda_val = model.params.iloc[1]

    if lambda_val >= 0:
        return np.nan

    half_life = -np.log(2) / lambda_val
    return round(half_life , 2)

# test on one pair 
beta, intercept = compute_hedge_ratio(train_prices , "MA" , "V")
spread = compute_spread(train_prices , "MA" , "V" , beta , intercept)
hl = compute_half_life(spread)
print(f"MA/V half-life : {hl} days")

MA/V half-life : 18.92 days


In [3]:
# compute Hurst exponent
def compute_hurst(spread , max_lag = 20):
    lags = range(2, max_lag)
    tau = []

    for lag in lags : 
        tau.append(np.std(np.subtract(spread[lag:].values,spread[:-lag].values)))

        # fit log - log regression
    log_lags = np.log(list(lags))
    log_tau = np.log(tau)

    hurst = np.polyfit(log_lags , log_tau , 1)[0]
    return round(hurst, 4)
""" 

Fit line:
hurst = np.polyfit(log_lags, log_tau, 1)[0]
np.polyfit(...,1) fits: y=mx+c
where:
x = log_lags
y = log_tau
The slope: m
is the Hurst exponent.

"""

# test on one pair 
beta , intercept = compute_hedge_ratio(train_prices , "MA" , "V")
spread = compute_spread(train_prices , "MA" , "V" , beta , intercept)
h = compute_hurst(spread)
print(f"MA/V Hurst exponent: {h}")
print(f"Interpretation: {'mean-reverting' if h < 0.5 else 'trending' if h > 0.5 else 'random walk'}")


MA/V Hurst exponent: 0.4288
Interpretation: mean-reverting


In [4]:
#  engineer all 8 features for every pair
def engineer_features(prices_period , ticker_1 , ticker_2):
    try :
        beta , intercept = compute_hedge_ratio(prices_period , ticker_1 , ticker_2)
        spread = compute_spread(prices_period , ticker_1 , ticker_2 ,  beta , intercept)

        # feature 1 - conintegration p-value 
        from statsmodels.tsa.stattools import coint
        _, coint_pval , _ = coint(prices_period[ticker_1] , prices_period[ticker_2]) 

        # feature 2 - correlation
        corr = prices_period[ticker_1].corr(prices_period[ticker_2])

        # feature 3 = spread volatility (std)
        spread_std = spread.std()

        # feature 4 - half life 
        hl = compute_half_life(spread)

        # feature 5 - hurest exponent
        hurst = compute_hurst(spread)

        # feature 6 - ADF p-value on spread
        adf_pval = adfuller(spread)[1]

        # feature 7 - spread skewness
        skew = spread.skew()

        # feature 8 - beta statbility 
        # compare beta from first half vs second half of period
        mid = len(prices_period) // 2
        beta_h1 , _ = compute_hedge_ratio(prices_period.iloc[:mid] , ticker_1 , ticker_2) 
        beta_h2 , _ = compute_hedge_ratio(prices_period.iloc[mid:] , ticker_1 , ticker_2)
        beta_stability  = abs(beta_h1 - beta_h2) / (abs(beta_h1) + 1e-6)

        return {
            "ticker_1" : ticker_1,
            "ticker_2" : ticker_2,
            "coint_pval" : round(coint_pval , 6) ,
            "correlation" : round(corr,4),
            "spread_std" : round(spread_std ,4),
            "half_life" : hl if not np.isnan(hl) else 999,
            "hurst" : hurst,
            "adf_pval" : round(adf_pval , ),
            "spread_skew" : round(skew , 4), 
            "beta_stability" : round(beta_stability , 4) 
        }
    
    except Exception as e :
        return None
    
# test in  one pair first 
features = engineer_features(train_prices , "MA" , "WMT")
print("Features for MA/WMT :")
for k,v in features.items():
    print(f" {k} : {v}")

Features for MA/WMT :
 ticker_1 : MA
 ticker_2 : WMT
 coint_pval : 0.003089
 correlation : 0.9282
 spread_std : 24.3726
 half_life : 20.55
 hurst : 0.4373
 adf_pval : 0
 spread_skew : -0.2467
 beta_stability : 0.3239


In [5]:
all_features = []

for _, row in train_pairs.iterrows():
    t1, t2 = row["ticker_1"], row["ticker_2"]
    features = engineer_features(train_prices, t1, t2)
    if features:
        all_features.append(features)
    print(f"Done: {t1}/{t2}")

features_df = pd.DataFrame(all_features)
print(f"\nFeatures computed for {len(features_df)} pairs")
print(features_df.head())

Done: META/TGT
Done: MA/WMT
Done: V/WMT
Done: GOOGL/UNH
Done: JNJ/TGT
Done: ABT/JNJ
Done: CL/WMT
Done: JNJ/MSFT
Done: KR/UNH
Done: C/GE
Done: AMZN/JNJ
Done: JNJ/UNH
Done: AMZN/CL
Done: CVX/WFC
Done: GIS/MA
Done: CAT/DE
Done: JNJ/KR
Done: MA/V
Done: ABT/CL
Done: JNJ/PYPL
Done: CL/V

Features computed for 21 pairs
  ticker_1 ticker_2  coint_pval  correlation  spread_std  half_life   hurst  \
0     META      TGT    0.001777       0.9592     18.5025      18.66  0.3924   
1       MA      WMT    0.003089       0.9282     24.3726      20.55  0.4373   
2        V      WMT    0.004086       0.9271     13.2146      22.10  0.4438   
3    GOOGL      UNH    0.006042       0.9654      7.5579      24.37  0.4422   
4      JNJ      TGT    0.006235       0.9318      5.5759      18.93  0.4581   

   adf_pval  spread_skew  beta_stability  
0         0       0.0007          0.8400  
1         0      -0.2467          0.3239  
2         0      -0.2048          0.3018  
3         0       0.3628          3.486

In [10]:
from data_loader import run_strategy_on_period, sharpe_ratio 

labels  = []

for _, row in features_df.iterrows():
    t1,t2 = row["ticker_1"] , row["ticker_2"]

    # run strategy on TEST period = label based on out-of-sample performance 
    result = run_strategy_on_period(test_prices , t1,t2)

    if result : 
        # label 1 = good pair (positive test Sharpe) , 0 = bad pair
        label = 1 if result["sharpe"] > 0 else 0
        labels.append({
            "ticker_1" : t1,
            "ticker_2" : t2,
            "test_sharpe" : result["sharpe"],
            "test_return" : result["total_return_pct"],
            "label" : label
        })

    else : 
        label.append({
            "ticker_1" : t1 , 
            "ticker_2" : t2,
            "test_sharpe" : result["sharpe"],
            "test_return" : result["total_return_pct"],
            "label" : label
        })


labels_df = pd.DataFrame(labels)

print("Label distribution : ")
print(labels_df["label"].value_counts())
print(f"\n Good pairs: {labels_df[labels_df['label'] == 1][["ticker_1" , "ticker_2" , "test_sharpe"]]}")

Label distribution : 
label
1    11
0    10
Name: count, dtype: int64

 Good pairs:    ticker_1 ticker_2  test_sharpe
1        MA      WMT       0.3046
2         V      WMT       0.1832
3     GOOGL      UNH       0.6794
5       ABT      JNJ       0.9742
8        KR      UNH       0.4099
9         C       GE       0.5171
12     AMZN       CL       0.0252
15      CAT       DE       0.5344
16      JNJ       KR       0.1340
17       MA        V       0.9809
18      ABT       CL       1.3207


In [14]:
ml_dataset = features_df.merge(labels_df , on = ["ticker_1" , "ticker_2"])

print("ML dataset shape:" , ml_dataset.shape)
print("\n Full dataset:")
print(ml_dataset[["ticker_1" , "ticker_2" , "hurst" , "half_life" , "beta_stability" , "test_sharpe" , "label"]])

ml_dataset.to_csv("../data/ml_dataset.csv" , index = False)
print("\n Saved to ml_dataset.csv") 

ML dataset shape: (21, 13)

 Full dataset:
   ticker_1 ticker_2   hurst  half_life  beta_stability  test_sharpe  label
0      META      TGT  0.3924      18.66          0.8400      -1.3989      0
1        MA      WMT  0.4373      20.55          0.3239       0.3046      1
2         V      WMT  0.4438      22.10          0.3018       0.1832      1
3     GOOGL      UNH  0.4422      24.37          3.4867       0.6794      1
4       JNJ      TGT  0.4581      18.93          0.3347      -0.2039      0
5       ABT      JNJ  0.4219      25.54          0.3524       0.9742      1
6        CL      WMT  0.4688      23.02          0.8164      -0.7139      0
7       JNJ     MSFT  0.4440      21.52          0.2034      -1.0171      0
8        KR      UNH  0.3761      16.27          0.0094       0.4099      1
9         C       GE  0.4956      30.67          2.1896       0.5171      1
10     AMZN      JNJ  0.4394      28.03          2.7692      -0.1399      0
11      JNJ      UNH  0.4271      25.90      